In [19]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-3.5-turbo")
llm.invoke("Who is the first Prime Minister of India?")

AIMessage(content='Jawaharlal Nehru', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 16, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dd8Tus2UJ1SQdyXomc8NsqFQVjFUQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0628-7d6b-7e33-98fa-c8ff635e814c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 6, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [22]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    """ 
    Annotated[list, add_messages] means:

    The field is a list

    When a node returns {"messages": [...]}, LangGraph should append those messages to the existing list.
    """

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

In [23]:
builder = StateGraph(State)

builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [24]:
message = {'role': 'user', 'content': 'Who walked on the moon for the first time? Print only the name.'}
response = graph.invoke({"messages": [message]})

In [25]:
response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name.', additional_kwargs={}, response_metadata={}, id='39120def-5985-4a95-bd01-b402a4ec9bca'),
 AIMessage(content='Neil Armstrong', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 22, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dd8TwV06UeyA1gf5yiedMQb14t3Cg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0628-8591-7232-a3b7-9d42f01005f7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 2, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_tok

In [26]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit", "exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role":"user", "content":in_message}]
        }
    else:
        state['messages'].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: As of my knowledge cutoff date in September 2021, the captain of the Indian cricket team was Virat Kohli. However, please note that this information may have changed, so I recommend checking the latest updates from a reliable source.
Bot: As of September 2021, Virat Kohli has scored over 22,000 runs across all formats of international cricket. This includes runs scored in Test matches, One Day Internationals (ODIs), and Twenty20 Internationals (T20Is). He is one of the leading run-scorers in international cricket history.
